# テキスト → トークン化 → 数値列：完全実装ガイド

> **対象**: NLP/ディープラーニング学習者  
> **最終更新**: 2026-04-13  
> **所要時間**: 2～3時間（全実装例実行）  
> **前提知識**: Python基礎、NumPy基礎

---

## 📚 学習ロードマップ

1. **セクション1**: ライブラリのインポート
2. **セクション2**: テキスト正規化
3. **セクション3**: トークン化（複数方式）
4. **セクション4**: トークンID変換と語彙構築
5. **セクション5**: 埋め込みベクトル層
6. **セクション6**: シーケンス統合（プーリング）
7. **セクション7**: 完全な実装パイプライン
8. **セクション8**: 埋め込みの類似度計算
9. **セクション9**: プーリング方法の比較
10. **セクション10**: エンドツーエンドデモ
11. **セクション11**: 応用例とベストプラクティス

## 1. ライブラリのインポート

このノートブックで使用するライブラリをインポートします。

In [52]:
import unicodedata
import re
import json
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from typing import List, Dict, Tuple
from collections import Counter
import pandas as pd
from scipy.spatial.distance import pdist, squareform

print('✓ All imports successful')

✓ All imports successful


## 2. テキスト正規化

### 2.1 定義と必要性

テキスト正規化とは、入力テキストを統一的な形式に変換するプロセスです：

- **Unicode正規化**: 異なる表現の同じ文字を統一
- **小文字変換**: 大文字・小文字を統一
- **非可視文字除去**: 制御文字やスペースの正規化


### 2.2 実装

In [53]:
def normalize_text(text):
    """
    テキストの正規化処理
    - Unicode正規化
    - 小文字化
    - 制御文字除去
    - 複数スペース統一
    """
    text = unicodedata.normalize('NFC', text)
    text = text.lower()
    text = ''.join(char for char in text if unicodedata.category(char) != 'Cc')
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


# テスト
sample_texts = [
    "  自然言語処理は　楽しい！  ",
    "ＮＬＰ処理（テスト）",
    "Hello   WORLD!!!"
]

print("テキスト正規化の例：")
for text in sample_texts:
    normalized = normalize_text(text)
    print(f"元: {repr(text)} → {repr(normalized)}")

テキスト正規化の例：
元: '  自然言語処理は\u3000楽しい！  ' → '自然言語処理は 楽しい！'
元: 'ＮＬＰ処理（テスト）' → 'ｎｌｐ処理（テスト）'
元: 'Hello   WORLD!!!' → 'hello world!!!'


## 3. トークン化

### 3.1 定義とトークナイザータイプ

トークン化とは、テキストを個々の単語（トークン）に分割するプロセス：

- **正規表現**: 単純で高速
- **WordPiece（BERT）**: 形態素解析使用、より精密
- **SentencePiece**: サブワード化対応


### 3.2 正規表現トークナイザー

In [54]:
class SimpleRegexTokenizer:
    """正規表現を使った簡単なトークナイザー"""
    
    @staticmethod
    def tokenize(text):
        tokens = re.findall(r'\w+', text.lower())
        return tokens


tokenizer = SimpleRegexTokenizer()
test_text = "Hello world! This is a test."
tokens = tokenizer.tokenize(test_text)
print(f"トークン: {tokens}")

トークン: ['hello', 'world', 'this', 'is', 'a', 'test']


### 3.3 BERTトークナイザー

In [55]:
try:
    from transformers import BertTokenizer
    
    class BertWordPieceTokenizer:
        def __init__(self):
            self.tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
        
        def tokenize(self, text):
            return self.tokenizer.tokenize(text)
    
    print('✓ BERT tokenizer loaded')
except:
    print('⚠ BERT not installed')
    BertWordPieceTokenizer = SimpleRegexTokenizer

✓ BERT tokenizer loaded


## 4. 語彙（Vocabulary）構築

### 4.1 定義と役割

語彙（Vocabulary）は、テキスト内の一意な単語を集約したもの：

- **word → ID**: 単語を数値にマッピング
- **ID → word**: 数値を単語に逆変換
- **特殊トークン**: <PAD>, <UNK>, <START>, <END>


### 4.2 実装

In [56]:
class Vocabulary:
    def __init__(self, special_tokens=None):
        self.word2id = {}
        self.id2word = {}
        self.special_tokens = special_tokens or ['<PAD>', '<UNK>', '<START>', '<END>']
        for token in self.special_tokens:
            idx = len(self.word2id)
            self.word2id[token] = idx
            self.id2word[idx] = token
    
    def add_word(self, word):
        if word not in self.word2id:
            idx = len(self.word2id)
            self.word2id[word] = idx
            self.id2word[idx] = word
    
    def encode(self, tokens):
        return [self.word2id.get(t, self.word2id['<UNK>']) for t in tokens]
    
    def decode(self, ids):
        return [self.id2word[i] for i in ids if i in self.id2word]
    
    def __len__(self):
        return len(self.word2id)


vocab = Vocabulary()
for word in ['hello', 'world', 'python']:
    vocab.add_word(word)
print(f"語彙サイズ: {len(vocab)}")

語彙サイズ: 7


## 5. バッチ処理とパディング

### 5.1 パディングの必要性

ニューラルネットワークは固定サイズ入力を要求：

- **バッチ処理**: 複数テキストの同時処理
- **パディング**: 短いテキストをゼロで埋める
- **アテンションマスク**: パディング位置を除外


### 5.2 実装

In [57]:
class BatchProcessor:
    @staticmethod
    def pad_sequences(sequences, max_len=None, pad_id=0):
        if max_len is None:
            max_len = max(len(seq) for seq in sequences)
        
        padded = np.zeros((len(sequences), max_len), dtype=np.long)
        mask = np.zeros_like(padded)
        
        for i, seq in enumerate(sequences):
            length = min(len(seq), max_len)
            padded[i, :length] = seq[:length]
            mask[i, :length] = 1
        
        return padded, mask


sequences = [[1, 2, 3], [4, 5], [6, 7, 8, 9]]
padded, mask = BatchProcessor.pad_sequences(sequences)
print(f"パディング形状: {padded.shape}")

パディング形状: (3, 4)


## 6. 埋め込みベクトル層

### 6.1 埋め込みの概念

埋め込み層は、離散的なトークンIDを連続的なベクトル空間にマッピング：

- **word2vec**: スキップグラムとCBOW
- **GloVe**: グローバルベクトル
- **埋め込み層（NN）**: ニューラルネットワークで学習


### 6.2 実装

In [58]:
class EmbeddingLayer:
    def __init__(self, vocab_size, embedding_dim):
        self.embedding_layer = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
    
    def embed(self, token_ids):
        if isinstance(token_ids, np.ndarray):
            token_ids = torch.from_numpy(token_ids).long()
        elif not isinstance(token_ids, torch.Tensor):
            token_ids = torch.tensor(token_ids, dtype=torch.long)
        
        return self.embedding_layer(token_ids)


embed_layer = EmbeddingLayer(vocab_size=1000, embedding_dim=64)
dummy_ids = np.array([[1, 2, 3], [4, 5, 0]])
embeddings = embed_layer.embed(dummy_ids)
print(f"埋め込み形状: {embeddings.shape}")

埋め込み形状: torch.Size([2, 3, 64])


## 7. シーケンス統合（プーリング）

### 7.1 プーリング手法

複数のトークン埋め込みから1つのベクトル表現を得る方法：

- **Mean Pooling**: 平均値を計算
- **CLS Token**: 最初のトークン使用
- **Max Pooling**: 最大値を計算
- **Last Token**: 最後のトークン使用


### 7.2 実装

In [59]:
class SequencePooling:
    @staticmethod
    def mean_pooling(embeddings, mask):
        masked = embeddings * mask.unsqueeze(-1)
        sum_embeddings = masked.sum(dim=1)
        sum_mask = mask.sum(dim=1, keepdim=True).clamp(min=1e-9)
        mean_result = sum_embeddings / sum_mask
        return mean_result
    
    @staticmethod
    def cls_pooling(embeddings):
        return embeddings[:, 0, :]
    
    @staticmethod
    def max_pooling(embeddings, mask):
        masked = embeddings.clone()
        masked[mask == 0] = float('-inf')
        return masked.max(dim=1)[0]
    
    @staticmethod
    def last_pooling(embeddings, mask):
        batch_size = embeddings.size(0)
        sequence_lengths = mask.sum(dim=1) - 1
        last_hidden = embeddings[range(batch_size), sequence_lengths]
        return last_hidden


print('✓ SequencePooling defined')

✓ SequencePooling defined


## 8. 完全な実装パイプライン

### 8.1 パイプラインアーキテクチャ

エンドツーエンド処理フロー：

```
Text → Tokenize → Encode → Pad → Embed → Pool → Vector
```


### 8.2 実装

In [60]:
class TextToEmbeddingPipeline:
    def __init__(self, vocab, embedding_dim=64):
        self.vocab = vocab
        self.embedding_layer = EmbeddingLayer(len(vocab), embedding_dim)
        self.pooling = SequencePooling()
    
    def process(self, texts, tokenizer=None, pooling_method='mean'):
        if tokenizer is None:
            tokenizer = SimpleRegexTokenizer()
        
        all_tokens = [tokenizer.tokenize(text) for text in texts]
        token_ids = [self.vocab.encode(tokens) for tokens in all_tokens]
        padded, mask = BatchProcessor.pad_sequences(token_ids)
        embeddings = self.embedding_layer.embed(padded)
        
        mask_tensor = torch.from_numpy(mask).float()
        if pooling_method == 'mean':
            result = self.pooling.mean_pooling(embeddings, mask_tensor)
        elif pooling_method == 'cls':
            result = self.pooling.cls_pooling(embeddings)
        elif pooling_method == 'max':
            result = self.pooling.max_pooling(embeddings, mask_tensor)
        else:
            result = self.pooling.last_pooling(embeddings, mask_tensor)
        
        return result


print('✓ TextToEmbeddingPipeline defined')

✓ TextToEmbeddingPipeline defined


## 9. 埋め込みの類似度計算

### 9.1 類似度尺度

ベクトル間の類似度を測定：

- **コサイン類似度**: 方向の類似性 (-1～1)
- **ユークリッド距離**: 距離ベース
- **内積**: マグニチュード考慮


### 9.2 実装

In [61]:
def compute_similarity_matrix(embeddings):
    if isinstance(embeddings, torch.Tensor):
        embeddings = embeddings.detach().numpy()
    
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    normalized = embeddings / (norms + 1e-8)
    similarity = np.dot(normalized, normalized.T)
    return similarity


print('✓ compute_similarity_matrix defined')

✓ compute_similarity_matrix defined


## 10. エンドツーエンドデモ

### 10.1 完全な実行例

In [62]:
# サンプルテキスト
sample_texts = [
    'I love natural language processing',
    'NLP is fascinating',
    'Deep learning models are powerful'
]

# 語彙構築
vocab = Vocabulary()
tokenizer = SimpleRegexTokenizer()
all_tokens = [tokenizer.tokenize(text) for text in sample_texts]

for tokens in all_tokens:
    for token in tokens:
        vocab.add_word(token)

print(f'語彙サイズ: {len(vocab)}')

# パイプラインで処理
pipeline = TextToEmbeddingPipeline(vocab, embedding_dim=64)
embeddings = pipeline.process(sample_texts, tokenizer=tokenizer, pooling_method='mean')

print(f'埋め込み形状: {embeddings.shape}')

# 類似度計算
similarity = compute_similarity_matrix(embeddings)
print(f'\n類似度行列:\n{similarity}')

語彙サイズ: 17
埋め込み形状: torch.Size([3, 64])

類似度行列:
[[ 0.9999998  -0.05870187  0.05340328]
 [-0.05870187  0.99999994 -0.22184137]
 [ 0.05340328 -0.22184137  0.9999998 ]]


## 11. 応用例とベストプラクティス

### 11.1 推奨事項

実践的な推奨事項：

- **前処理**: 常にNorm化を実行
- **プーリング**: タスク依存で選択
- **語彙**: min_freqでノイズフィルタ
- **パディング**: 動的か固定を検討
- **評価**: 類似度相関を検証
